## Stage 1 — FAST (resume from epoch 15 → 19)
Run the backup cell first. The log should say `Starting from epoch 16` and `ENTERING TRAINING PHASE 2` with a fresh Phase 2 optimizer (the checkpoint is from Phase 1, so no scheduler restore). After a disconnect it says `Restored LR scheduler state from checkpoint`; run the same cell again to resume. If the stage is already finished it prints `nothing to train` and exits without touching your report.

In [ ]:
# 5b. BACKUP before resuming (each epoch overwrites last_epoch.pth). Safe to re-run.
import os, shutil
DRIVE = "/content/drive/MyDrive/AgriNexus"
CKPT = f"{DRIVE}/checkpoints"
assert os.path.isdir(CKPT), "Drive not mounted: run cell 3 first"
if not os.path.exists(f'{CKPT}/epoch15_final.pth') and os.path.exists(f'{CKPT}/last_epoch.pth'):
    shutil.copy(f'{CKPT}/last_epoch.pth', f'{CKPT}/epoch15_final.pth')
    print('Backed up last_epoch.pth -> epoch15_final.pth')
# evaluate.py needs best_model.pth; if the old one is missing, use epoch 15 as the baseline
if not os.path.exists(f'{CKPT}/best_model.pth') and os.path.exists(f'{CKPT}/epoch15_final.pth'):
    shutil.copy(f'{CKPT}/epoch15_final.pth', f'{CKPT}/best_model.pth')
    print('best_model.pth was missing -> seeded from epoch 15')
!ls -la {CKPT}

In [ ]:
# 1. GPU check (note which GPU you got: T4 / L4 / A100 changes epoch time a lot)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install libraries
!pip install -q "albumentations>=1.4.0" "torchmetrics>=1.4.0" "timm>=1.0.3" "scikit-learn>=1.5.0" pyyaml pillow tqdm

In [ ]:
# 3. Mount Google Drive (use your PERSONAL account)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/AgriNexus'
CKPT = f'{DRIVE}/checkpoints'
os.makedirs(CKPT, exist_ok=True)
!ls -la {DRIVE} {CKPT}

In [ ]:
# 4. Copy code + dataset to the fast local disk and unpack (dataset stays local for speed)
%cd /content
!cp -n {DRIVE}/agrinexus_code_v2.zip {DRIVE}/agrinexus_dataset.zip /content/
!unzip -q -o agrinexus_code_v2.zip
!test -d leaf_dataset || unzip -q -o agrinexus_dataset.zip
!ls ml/disease/config_*.yaml
!ls leaf_dataset | head

In [ ]:
# 5. Smoke test (1 mini-batch, writes to a throwaway dir - NOT to Drive)
!python ml/disease/train.py --config ml/disease/config_fast.yaml --output-dir /content/dry --dry-run --device cuda

## Stage 1 — FAST (resume from epoch 9 → 19)
Log should say `Starting from epoch 10` and `Fast-forwarded LR scheduler by 4 epoch(s)` on the first run. After a disconnect it says `Restored LR scheduler state from checkpoint` instead. Run this same cell again to resume.

In [ ]:
# 6. STAGE 1 - FAST
!python ml/disease/train.py \
  --config ml/disease/config_fast.yaml \
  --output-dir {CKPT} \
  --resume {CKPT}/last_epoch.pth \
  --device cuda

In [ ]:
# 7. Evaluate the best Stage 1 model
!python ml/disease/evaluate.py \
  --checkpoint {CKPT}/best_model.pth \
  --gate1-checkpoint {CKPT}/gate1_model.pth \
  --config ml/disease/config_fast.yaml \
  --device cuda

## Stage 2 — FULL (run only after Stage 1 finished, epoch 19 → 50)
First launch uses `--reset-scheduler` (fresh warm-restart cycle). **After a disconnect, use the second cell (no flag)** or the schedule would restart again.

In [ ]:
# 8a. STAGE 2 - FIRST launch only
!python ml/disease/train.py \
  --config ml/disease/config_full.yaml \
  --output-dir {CKPT} \
  --resume {CKPT}/last_epoch.pth \
  --reset-scheduler \
  --device cuda

In [ ]:
# 8b. STAGE 2 - RESUME after a disconnect (no --reset-scheduler)
!python ml/disease/train.py \
  --config ml/disease/config_full.yaml \
  --output-dir {CKPT} \
  --resume {CKPT}/last_epoch.pth \
  --device cuda